   Phase 1 corpus collection (cs.CL, 2023–2025). Builds corpus.csv (400 papers) and collection_info.json.
   The search engine and GUI are in app.py (run: streamlit run app.py).
  

In [18]:
!pip install -q arxiv pypdf

In [ ]:
import arxiv
import json
import time
import urllib.request
import pandas as pd
from pathlib import Path
from pypdf import PdfReader

CORPUS = "corpus.csv"
Path("pdfs").mkdir(exist_ok=True)

topics = {
    "Machine Translation": '"machine translation"',
    "Question Answering": '"question answering"',
    "Summarization": "summarization",
    "Dialogue Systems": '"dialogue systems"'
}

# 12 quarters (2023-Q1 ... 2025-Q4). 100 papers per topic, spread evenly:
# 9 per quarter in 2023, 8 per quarter in 2024-2025  ->  4*9 + 8*8 = 100
quarters = []
for year in (2023, 2024, 2025):
    for q, (m1, m2, last_day) in enumerate([(1, 3, 31), (4, 6, 30), (7, 9, 30), (10, 12, 31)], 1):
        quarters.append({
            "name": f"{year}-Q{q}",
            "date": f"submittedDate:[{year}{m1:02d}010000 TO {year}{m2:02d}{last_day}2359]",
            "id_months": {f"{year % 100:02d}{m:02d}" for m in range(m1, m2 + 1)},
            "quota": 9 if year == 2023 else 8,
        })

# Resume: load what was already collected (delete corpus.csv to start over)
if Path(CORPUS).exists():
    data = pd.read_csv(CORPUS, dtype={"arxiv_id": str}).fillna("").to_dict("records")
else:
    data = []
seen = {str(r["arxiv_id"]).split("v")[0] for r in data}
queries = {}

client = arxiv.Client(page_size=100, delay_seconds=3, num_retries=5)


def save():
    pd.DataFrame(data).to_csv(CORPUS, index=False, encoding="utf-8")


for topic, keyword in topics.items():
    for qt in quarters:
        query = f"cat:cs.CL AND {keyword} AND {qt['date']}"
        queries[f"{topic} | {qt['name']}"] = query

        have = sum(1 for r in data if r["sub_topic"] == topic and r["period"] == qt["name"])
        need = qt["quota"]
        if have >= need:
            continue
        print(f"Starting {topic} {qt['name']}: {have}/{need}")

        search = arxiv.Search(query=query, max_results=300,
                              sort_by=arxiv.SortCriterion.SubmittedDate)
        try:
            for paper in client.results(search):
                if have >= need:
                    break
                paper_id = paper.get_short_id()
                base_id = paper_id.split("v")[0]

                if base_id in seen:
                    continue
                if paper.primary_category != "cs.CL":      # truly in our category
                    continue
                if base_id[:4] not in qt["id_months"]:     # ID year-month must be in this quarter
                    continue

                path = Path("pdfs") / f"{base_id}.pdf"
                try:
                    if not path.exists():
                        urllib.request.urlretrieve(paper.pdf_url, str(path))
                        time.sleep(1)

                    reader = PdfReader(str(path))
                    pages = len(reader.pages)
                    if pages == 6:                         # 6 pages fits no tier
                        seen.add(base_id)
                        continue

                    content = "\n".join((p.extract_text() or "") for p in reader.pages)
                    content = content.encode("utf-8", errors="replace").decode("utf-8")
                    if len(content.strip()) < 1000:        # unreadable / scanned PDF
                        seen.add(base_id)
                        continue

                    length = "Short" if pages < 6 else ("Medium" if pages <= 9 else "Long")

                    data.append({
                        "title": " ".join(paper.title.split()),
                        "content": content,
                        "arxiv_id": paper_id,
                        "arxiv_link": paper.entry_id,
                        "sub_topic": topic,
                        "pages": pages,
                        "length": length,
                        "published": paper.published.date().isoformat(),
                        "period": qt["name"],
                    })
                    seen.add(base_id)
                    have += 1
                    save()
                    print(f"  {topic} {qt['name']} {have}/{need}  ({pages} pages, {length})")

                except Exception as e:
                    print("  Skipping", paper_id, ":", e)
                    time.sleep(2)

        except Exception as e:
            print("  Search interrupted:", e, "-> run this cell again to resume")

with open("collection_info.json", "w", encoding="utf-8") as f:
    json.dump({
        "category": "cs.CL",
        "date_range": "2023-01-01 to 2025-12-31",
        "sampling": "100 papers per sub-topic spread over 12 quarters "
                    "(9 per quarter in 2023, 8 per quarter in 2024-2025); "
                    "within each quarter, newest first",
        "filters": [
            "primary category is cs.CL",
            "arXiv ID year-month inside the queried quarter",
            "no duplicates across sub-topics",
            "exactly 6 pages excluded (fits no length tier)",
            "less than 1000 characters of extractable text excluded",
        ],
        "queries": queries,
    }, f, indent=2)

df = pd.DataFrame(data)
print("\nTotal papers:", len(df))
print("\nBy sub-topic:\n", df["sub_topic"].value_counts())
print("\nBy length:\n", df["length"].value_counts())
print("\nBy year:\n", df["period"].str[:4].value_counts().sort_index())
print("\nSub-topic x length:\n", pd.crosstab(df["sub_topic"], df["length"]))
print("\nIDs outside 2301-2512:",
      (~df["arxiv_id"].str[:4].between("2301", "2512")).sum())


Starting Dialogue Systems 2023-Q1: 0/9
  Dialogue Systems 2023-Q1 1/9  (15 pages, Long)
  Dialogue Systems 2023-Q1 2/9  (14 pages, Long)
  Dialogue Systems 2023-Q1 3/9  (27 pages, Long)
  Dialogue Systems 2023-Q1 4/9  (5 pages, Short)
  Dialogue Systems 2023-Q1 5/9  (9 pages, Medium)


  Dialogue Systems 2023-Q1 6/9  (18 pages, Long)
  Dialogue Systems 2023-Q1 7/9  (8 pages, Medium)
  Dialogue Systems 2023-Q1 8/9  (9 pages, Medium)
  Dialogue Systems 2023-Q1 9/9  (12 pages, Long)
Starting Dialogue Systems 2023-Q2: 0/9
  Dialogue Systems 2023-Q2 1/9  (14 pages, Long)
  Dialogue Systems 2023-Q2 2/9  (15 pages, Long)
  Dialogue Systems 2023-Q2 3/9  (5 pages, Short)
  Dialogue Systems 2023-Q2 4/9  (17 pages, Long)
  Dialogue Systems 2023-Q2 5/9  (13 pages, Long)
  Dialogue Systems 2023-Q2 6/9  (12 pages, Long)
  Dialogue Systems 2023-Q2 7/9  (11 pages, Long)
  Dialogue Systems 2023-Q2 8/9  (8 pages, Medium)
  Dialogue Systems 2023-Q2 9/9  (17 pages, Long)
Starting Dialogue Systems 2023-Q3: 0/9
  Dialogue Systems 2023-Q3 1/9  (11 pages, Long)
  Skipping 2309.13243v2 : HTTP Error 404: Not Found
  Dialogue Systems 2023-Q3 2/9  (8 pages, Medium)
  Dialogue Systems 2023-Q3 3/9  (16 pages, Long)
  Skipping 2309.09749v3 : HTTP Error 404: Not Found
  Dialogue Systems 2023-Q3 4/9

In [19]:
import pandas as pd
df = pd.read_csv("corpus.csv", dtype={"arxiv_id": str})

print("Total papers:", len(df))
print("\nBy sub-topic:\n", df["sub_topic"].value_counts())
print("\nBy length:\n", df["length"].value_counts())
print("\nBy year:\n", df["period"].str[:4].value_counts().sort_index())
print("\nSub-topic x length:\n", pd.crosstab(df["sub_topic"], df["length"], margins=True))
print("\nPages: min", df["pages"].min(), "| median", df["pages"].median(), "| max", df["pages"].max())
display(df[["title", "arxiv_id", "sub_topic", "pages", "length", "period"]].head())

Total papers: 400

By sub-topic:
 sub_topic
Machine Translation    100
Question Answering     100
Summarization          100
Dialogue Systems       100
Name: count, dtype: int64

By length:
 length
Long      308
Medium     66
Short      26
Name: count, dtype: int64

By year:
 period
2023    144
2024    128
2025    128
Name: count, dtype: int64

Sub-topic x length:
 length               Long  Medium  Short  All
sub_topic                                    
Dialogue Systems       71      21      8  100
Machine Translation    70      21      9  100
Question Answering     84      11      5  100
Summarization          83      13      4  100
All                   308      66     26  400

Pages: min 2 | median 15.0 | max 144


,title,arxiv_id,sub_topic,pages,length,period
0,Exploiting Multilingualism in Low-resource Neu...,2303.18011v1,Machine Translation,10,Long,2023-Q1
1,$\varepsilon$ KÚ <MASK>: Integrating Yorùbá cu...,2303.17972v2,Machine Translation,7,Medium,2023-Q1
2,Selective Knowledge Distillation for Non-Autor...,2303.17910v2,Machine Translation,9,Medium,2023-Q1
3,Reference-less Analysis of Context Specificity...,2303.16618v3,Machine Translation,16,Long,2023-Q1
4,Hallucinations in Large Multilingual Translati...,2303.16104v1,Machine Translation,32,Long,2023-Q1


In [20]:
df["id_month"] = df["arxiv_id"].str[:4]
print(df["id_month"].value_counts().sort_index())
print("Outside 2023-2025:", (~df["id_month"].between("2301", "2512")).sum())

id_month
2303    36
2306    36
2309    36
2312    36
2403    32
2406    32
2409    32
2412    32
2503    32
2506    32
2509    32
2512    32
Name: count, dtype: int64
Outside 2023-2025: 0
